# 03 – Privacy & Governance Analysis
**Role:** Governance Officer  
**Project:** NovaCred Credit Application Governance Audit  
**Course:** DEGO 2606 – Nova SBE  

Covers:
1. PII Identification
2. Pseudonymization & Data Minimization
3. GDPR Compliance Gap Analysis (specific Articles)
4. EU AI Act Classification
5. Actionable Governance Controls

**Input:** `../data/credit_applications_clean_final.csv` (498 records, produced by `01-data-quality.ipynb`)


In [3]:
import pandas as pd
import numpy as np
import hashlib
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DATA_PATH = Path("../data/credit_applications_clean_final.csv")
df = pd.read_csv(DATA_PATH)
df["applicant_info.date_of_birth"] = pd.to_datetime(
    df["applicant_info.date_of_birth"], errors="coerce"
)

print(f"Dataset loaded: {len(df)} records, {len(df.columns)} columns")
df.head(3)


Dataset loaded: 498 records, 22 columns


,_id,spending_behavior,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,...,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes,_missing_critical_financials
0,app_200,"[{'category': 'Shopping', 'amount': 480}, {'ca...",2024-01-15 00:00:00+00:00,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036.0,...,0.20,31212,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0
1,app_037,"[{'category': 'Rent', 'amount': 608}, {'catego...",NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,Male,1992-03-31,10032.0,...,0.18,17915,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0
2,app_215,"[{'category': 'Rent', 'amount': 109}]",NaN,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075.0,...,0.21,37909,True,NaN,vacation,3.7,59000.0,NaN,NaN,0


## 1. PII Identification

**GDPR Art. 4(1):** Personal data = any information relating to an identified or 
identifiable natural person.

Cross-notebook evidence already established:
- **Notebook 01:** 3 SSNs shared across different applicants — accuracy violation (Art. 5(1)(d))
- **Notebook 02:** ZIP code confirmed as proxy for gender — female-majority region has lowest approval rate (52%)


In [4]:
# Section 1: PII Inventory
# Identifies all personal data fields under GDPR Art. 4(1)

pii_inventory = [
    {"Column": "applicant_info.full_name",    "PII Type": "Direct PII",    "GDPR Article": "Art. 4(1)",           "Risk": "High",     "Finding": "Present in 100% of records; plain text; enables direct identification"},
    {"Column": "applicant_info.email",         "PII Type": "Direct PII",    "GDPR Article": "Art. 4(1)",           "Risk": "High",     "Finding": "Present in 100% of records; unique identifier"},
    {"Column": "applicant_info.ssn",           "PII Type": "Direct PII",    "GDPR Article": "Art. 87",             "Risk": "Critical", "Finding": "3 SSNs shared across different people — accuracy breach (notebook 01)"},
    {"Column": "applicant_info.ip_address",    "PII Type": "Indirect PII",  "GDPR Article": "Art. 4(1) Recital 30","Risk": "Medium",   "Finding": "Enables geolocation and device fingerprinting; missing in 4 records"},
    {"Column": "applicant_info.date_of_birth", "PII Type": "Direct PII",    "GDPR Article": "Art. 4(1)",           "Risk": "High",     "Finding": "Combined with name = strong re-identification risk; missing in 4 records"},
    {"Column": "applicant_info.gender",        "PII Type": "Direct PII",    "GDPR Article": "Art. 4(1) / Art. 9", "Risk": "High",     "Finding": "Used in lending model; DI ratio = 0.767 (notebook 02)"},
    {"Column": "applicant_info.zip_code",      "PII Type": "Indirect PII",  "GDPR Article": "Art. 4(1)",           "Risk": "Medium",   "Finding": "Confirmed proxy for gender in notebook 02"},
    {"Column": "spending_behavior",            "PII Type": "Behavioural PII","GDPR Article": "Art. 5(1)(c)",       "Risk": "High",     "Finding": "Granular spending categories build sensitive profile; necessity not demonstrated"},
]

pii_df = pd.DataFrame(pii_inventory)
print(f"=== PII Fields Identified: {len(pii_df)} total ===\n")
print(pii_df[["Column", "PII Type", "Risk", "GDPR Article"]].to_string(index=False))
print(f"\nRisk breakdown:\n{pii_df['Risk'].value_counts().to_string()}")


=== PII Fields Identified: 8 total ===

                      Column        PII Type     Risk         GDPR Article
    applicant_info.full_name      Direct PII     High            Art. 4(1)
        applicant_info.email      Direct PII     High            Art. 4(1)
          applicant_info.ssn      Direct PII Critical              Art. 87
   applicant_info.ip_address    Indirect PII   Medium Art. 4(1) Recital 30
applicant_info.date_of_birth      Direct PII     High            Art. 4(1)
       applicant_info.gender      Direct PII     High   Art. 4(1) / Art. 9
     applicant_info.zip_code    Indirect PII   Medium            Art. 4(1)
           spending_behavior Behavioural PII     High         Art. 5(1)(c)

Risk breakdown:
Risk
High        5
Medium      2
Critical    1


## 2. Pseudonymization & Data Minimization

**GDPR Art. 4(5):** Pseudonymisation = processing so data can no longer be attributed 
to a specific person without additional information kept separately.

Two techniques demonstrated:
- **SHA-256 hashing** on 4 direct PII fields (name, email, SSN, IP address)
- **Age bracketing** on `date_of_birth` — implements data minimisation (Art. 5(1)(c))

> Note: Pseudonymized data is still personal data under GDPR (Recital 26).  
> Re-identification is possible with the original values.


In [5]:
# SHA-256 pseudonymization of direct PII fields (GDPR Art. 4(5))
# One-way hash: same input always gives same hash, but original cannot be recovered

def sha256_hash(value):
    """SHA-256 one-way hash. Returns None if value is missing."""
    if pd.isna(value) or value is None:
        return None
    return hashlib.sha256(str(value).strip().encode("utf-8")).hexdigest()

fields_to_pseudonymize = [
    "applicant_info.full_name",
    "applicant_info.email",
    "applicant_info.ssn",
    "applicant_info.ip_address",
]

df_pseudo = df.copy()

for field in fields_to_pseudonymize:
    if field in df_pseudo.columns:
        df_pseudo[f"{field}_pseudo"] = df_pseudo[field].apply(sha256_hash)
        df_pseudo.drop(columns=[field], inplace=True)
        print(f"  ✓ '{field}' pseudonymized")

# Show before vs after
print("\n--- BEFORE (first 3 records) ---")
print(df[["_id", "applicant_info.full_name", "applicant_info.ssn"]].head(3).to_string(index=False))

print("\n--- AFTER pseudonymization (first 3 records) ---")
print(df_pseudo[["_id", "applicant_info.full_name_pseudo", "applicant_info.ssn_pseudo"]].head(3).to_string(index=False))


  ✓ 'applicant_info.full_name' pseudonymized
  ✓ 'applicant_info.email' pseudonymized
  ✓ 'applicant_info.ssn' pseudonymized
  ✓ 'applicant_info.ip_address' pseudonymized

--- BEFORE (first 3 records) ---
    _id applicant_info.full_name applicant_info.ssn
app_200              Jerry Smith        596-64-4340
app_037           Brandon Walker        425-69-4784
app_215              Scott Moore        370-78-5178

--- AFTER pseudonymization (first 3 records) ---
    _id                                  applicant_info.full_name_pseudo                                        applicant_info.ssn_pseudo
app_200 68ee17cf46b0560352c701bbbdb178c01d4cc368014d188113194cd88e13fa96 2caf30528c21a10e1307b27f9dbbfc312f0c00d46b333ef8ddabb44b2511ccbd
app_037 4c539f3c4c8794d5d0b7ef2d2c6f3e5ae3e9ab65f1cb54d1de80a25fbe82bb70 2f7da45fefdcfb2c5b4f5b6f1465c22054c36e04fc77c181341c5163d64405ac
app_215 4ad1a6eb65ea21350865c8dbae97a882eb82ee36059784840bd215ff2959a355 db120edcee2366a48d6d77c2db8c64c5536b8dc3c3c524f2e1